In [1]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge
import xgboost as xgb
import lightgbm as lgb
from sklearn.svm import SVR
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, root_mean_squared_error, mean_absolute_error
import joblib 

#
df = pd.read_csv("Ames_Housing_Processed.csv")

X = df.drop('SalePrice', axis=1)
y = df['SalePrice']


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

level0_models = [
    ('lgbm', lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, num_leaves=15, max_depth=6)),
    ('xgb', xgb.XGBRegressor(n_estimators=1500, learning_rate=0.05, max_depth=3, colsample_bytree=0.6)),
    ('svr', SVR(kernel='poly', C=10, epsilon=0.1))
]


level1_model = Ridge(alpha=1.0)


stacking_regressor = StackingRegressor(
    estimators = level0_models,
    final_estimator=level1_model,
    cv=5,
    n_jobs=-1
)


stacking_regressor.fit(X_train, y_train)


predictions = stacking_regressor.predict(X_test)


r2 = r2_score(y_test, predictions)
mae = mean_absolute_error(y_test, predictions)
rmse = root_mean_squared_error(y_test, predictions)

print(f"R2: {r2}")
print(f"MAE: {mae}")
print(f"RMSE: {rmse}")
print(f"Model Weights (LGBM, XGB, SVR): {stacking_regressor.final_estimator_.coef_}")


model_filename = "ames_housing_stacking_model.pkl"
joblib.dump(stacking_regressor, model_filename)

print(f"\n[Transaction complete.]: {model_filename}")


R2: 0.9108066442728066
MAE: 13233.241637192115
RMSE: 20483.226402861383
Model Weights (LGBM, XGB, SVR): [ 0.06911577  0.94116665 -0.10014661]

[Transaction complete.]: ames_housing_stacking_model.pkl
